In [1]:
# ----------------------------
# Bank System — Day 5
# Audit & Risk Analysis Demo
# ----------------------------

import sys

from audit.enums import RiskLevel

sys.path.append("../src")

from datetime import datetime, timedelta, date
from collections import Counter

from bank.bank import Bank
from bank.client import Client, Contact

from account.bank_account import BankAccount
from account.savings_account import SavingsAccount
from account.premium_account import PremiumAccount

from account.enums import AccountCurrency

from transaction.transaction import Transaction
from transaction.transaction_queue import TransactionAlreadyInQueue, QueueIsEmpty


# ----------------------------
# Create bank
# ----------------------------

bank = Bank()
print("Bank created")


# ----------------------------
# Create clients
# ----------------------------

clients = [
    Client(
        name="Иван Иванов",
        client_id="c1",
        date_of_birth=date(1993, 2, 15),
        login="ivan",
        password="1234",
        contacts=[Contact("email", "ivan@mail.com")]
    ),

    Client(
        name="Мария Петрова",
        client_id="c2",
        date_of_birth=date(1998, 7, 20),
        login="maria",
        password="abcd",
        contacts=[Contact("phone", "+79991234567")]
    ),

    Client(
        name="Сергей Сидоров",
        client_id="c3",
        date_of_birth=date(1983, 11, 5),
        login="sergey",
        password="qwerty",
        contacts=[Contact("email", "sergey@mail.com")]
    ),
]

for c in clients:
    bank.add_client(c)

print("Clients:", [c.name for c in bank.clients])


# ----------------------------
# Create accounts
# ----------------------------

ivan, maria, sergey = clients

ivan_acc1 = BankAccount(owner_name=ivan.name)
ivan_acc2 = SavingsAccount(owner_name=ivan.name, min_balance=100, monthly_interest=1)

maria_acc1 = PremiumAccount(owner_name=maria.name, fee=5)

sergey_acc1 = BankAccount(owner_name=sergey.name)

bank.open_account(ivan.client_id, ivan_acc1)
bank.open_account(ivan.client_id, ivan_acc2)

bank.open_account(maria.client_id, maria_acc1)

bank.open_account(sergey.client_id, sergey_acc1)

print("Accounts created")


# ----------------------------
# Initial balances
# ----------------------------

ivan_acc1.deposit(20000)
ivan_acc2.deposit(5000)

maria_acc1.deposit(10000)

sergey_acc1.deposit(1000)

print("\nInitial balances:")
print("Ivan:", ivan_acc1.balance, ivan_acc2.balance)
print("Maria:", maria_acc1.balance)
print("Sergey:", sergey_acc1.balance)


# ----------------------------
# Create transactions (expanded to ~20)
# ----------------------------

queue = bank.transaction_queue

transactions = [
    # обычные переводы
    Transaction(1000, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(2000, AccountCurrency.USD, 1, ivan_acc1, sergey_acc1),
    Transaction(300, AccountCurrency.USD, 1, maria_acc1, ivan_acc1),
    Transaction(1500, AccountCurrency.USD, 1, ivan_acc2, maria_acc1),

    # отложенные операции
    Transaction(500, AccountCurrency.USD, 1, maria_acc1, sergey_acc1, execute_time=datetime.now() + timedelta(seconds=10)),
    Transaction(700, AccountCurrency.USD, 1, ivan_acc1, maria_acc1, execute_time=datetime.now() + timedelta(seconds=5)),

    # мелкие частые операции (для frequent)
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),

    # крупные суммы
    Transaction(9000, AccountCurrency.USD, 1, ivan_acc1, maria_acc1),
    Transaction(12000, AccountCurrency.USD, 1, maria_acc1, ivan_acc1),
]

BankAccount_new = BankAccount(owner_name="Test Client")
# создаём ещё пару новых счетов для новых клиентов
new_acc1 = BankAccount(owner_name="New Client 1")
new_acc2 = BankAccount(owner_name="New Client 2")

bank.open_account(ivan.client_id, new_acc1)
bank.open_account(maria.client_id, new_acc2)

transactions += [
    Transaction(500, AccountCurrency.USD, 1, ivan_acc1, new_acc1),
    Transaction(600, AccountCurrency.USD, 1, maria_acc1, new_acc2),
    Transaction(700, AccountCurrency.USD, 1, ivan_acc1, new_acc2),
    Transaction(400, AccountCurrency.USD, 1, maria_acc1, new_acc1),
]

# транзакция на замороженный счет
bank.freeze_account(sergey_acc1.id)
transactions += [
    Transaction(200, AccountCurrency.USD, 1, ivan_acc1, sergey_acc1),
]

# добавляем все транзакции в очередь
for t in transactions:
    try:
        queue.enqueue(t)
    except TransactionAlreadyInQueue:
        pass

print(f"{len(transactions)} transactions added to queue")


# ----------------------------
# Suspicious scenarios
# ----------------------------

# frequent transactions
for _ in range(6):
    queue.enqueue(
        Transaction(
            200,
            AccountCurrency.USD,
            1,
            ivan_acc1,
            maria_acc1
        )
    )

# large amount
queue.enqueue(
    Transaction(
        9000,
        AccountCurrency.USD,
        1,
        ivan_acc1,
        maria_acc1
    )
)

print("Suspicious transactions added")


# ----------------------------
# Freeze account scenario
# ----------------------------

bank.freeze_account(sergey_acc1.id)
print("Sergey account frozen")


# ----------------------------
# Duplicate transaction test
# ----------------------------

try:
    queue.enqueue(transactions[0])
except TransactionAlreadyInQueue:
    print("Duplicate transaction detected")


# ----------------------------
# Process transaction queue
# ----------------------------

processor = bank.transaction_processor

print("\nProcessing queue...")

while True:
    try:
        processor.process()
    except QueueIsEmpty:
        print("Queue is empty")
        break


# ----------------------------
# Transaction results
# ----------------------------

print("\nTransaction results")

for t in transactions:

    reject = None

    if t.reject_reason:
        reject = f"{type(t.reject_reason).__name__}: {t.reject_reason}"

    print(
        t.transaction_id,
        t.status.name,
        "reject:",
        reject
    )


# ----------------------------
# Risk analyzer demo
# ----------------------------

print("\nRisk analysis example")

risk = bank.risk_analyzer.analyze_transaction(transactions[0], ivan)
print("Transaction risk:", risk)


# ----------------------------
# Suspicious transactions report
# ----------------------------

print("\nSuspicious operations")

suspicious = bank.audit_log.get_suspicious_transactions()

for log in suspicious:
    print(
        getattr(log, 'client_id', ' - '),
        getattr(log, 'risk', ' - '),
        getattr(log, 'created_at', ' - '),
    )


# ----------------------------
# Client risk profile
# ----------------------------

print("\nIvan risk profile")

profile = bank.audit_log.get_client_risk_profile("c1")
print(f"LOW: {profile.get(RiskLevel.LOW, "-")}")
print(f"MEDIUM: {profile.get(RiskLevel.MEDIUM, "-")}")
print(f"HIGH: {profile.get(RiskLevel.HIGH, "-")}")

# ----------------------------
# Error statistics
# ----------------------------

print("\nError statistics")

errors = bank.audit_log.get_errors()

for error in errors:
    print(error)

# ----------------------------
# Final balances
# ----------------------------

print("\nFinal balances")

print("Ivan acc1:", ivan_acc1.balance)
print("Ivan acc2:", ivan_acc2.balance)

print("Maria:", maria_acc1.balance)

print("Sergey:", sergey_acc1.balance)

bank.audit_log.save_logs_to_json_file("logs.json")

Bank created
Clients: ['Иван Иванов', 'Мария Петрова', 'Сергей Сидоров']
Accounts created

Initial balances:
Ivan: 20000 5000
Maria: 10000
Sergey: 1000
19 transactions added to queue
Suspicious transactions added
Sergey account frozen
Duplicate transaction detected

Processing queue...
Queue is empty

Transaction results
ff0100ce-230c-4bc0-baf2-bb8e9f21877c EXECUTED reject: None
f00cfa7d-6fcc-4890-a7a4-19f60527f886 REJECTED reject: AccountFrozenError: 
b7bb0b62-b8d9-4787-9f8a-d62a08d7f618 EXECUTED reject: None
0beb10d9-0f7c-4977-96db-4589bd485c53 EXECUTED reject: None
cdbd7807-c66e-4f42-aa8c-fd58eaa11574 CREATED reject: None
558b11c2-e210-4fd4-ab91-5197ac5a89bd CREATED reject: None
c125e584-c7e0-4a65-aaa7-2ba61e00d3d8 EXECUTED reject: None
6bafe85b-d352-43b4-9cef-2f2829e6fa40 EXECUTED reject: None
0728e3af-c4d5-4888-b520-5510b076041f EXECUTED reject: None
32d8c9fc-a1db-4661-931d-5727ab8c5f70 CANCELED reject: None
4bed4fbc-624d-4878-8526-8747d1313028 CANCELED reject: None
996ec2c5-b24e-